# Recurrent Neural Networks (RNNs) - TensorFlow/Keras

Recurrent Neural Networks (RNNs) are a class of neural networks built for **sequential data**. Unlike feed-forward networks, an RNN carries a "memory": a hidden state that summarises everything it has read so far.

At each time step $t$ the network combines the current input $x_t$ with the previous hidden state $h_{t-1}$:

$h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t + b_h)$

The same weights are reused at every step, which is what lets one model handle sequences of any length:

```mermaid
flowchart LR
    x1[Input x1] --> h1[Hidden state h1]
    x2[Input x2] --> h2[Hidden state h2]
    x3[Input x3] --> h3[Hidden state h3]
    h1 --> h2
    h2 --> h3
    h3 --> out[Prediction]
```

A feed-forward network processes each input independently and so cannot use this context. That makes RNNs a natural fit for tasks where order carries meaning:

- Speech recognition
- Sentiment analysis
- Time series forecasting
- Machine translation

## Learning Objectives

At the end of this notebook, you should be able to:

* Build and train SimpleRNN, LSTM and GRU models in TensorFlow/Keras
* Measure how far a gradient travels back through time, and explain when it vanishes or explodes
* Compare the three architectures by parameter cost and by the control each one gives over gradient flow
* Apply an LSTM to a time series forecast and evaluate it against a baseline with MAE and RMSE

## Setup and Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping

RSEED = 42
np.random.seed(RSEED)
tf.random.set_seed(RSEED)

print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["figure.titlesize"] = 14
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["figure.figsize"] = (10, 6)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU available: {len(gpus)} GPU(s)")
    for gpu in gpus:
        print(f"  - {gpu}")
else:
    print("No GPU found, using CPU")

## Part 1: Simple RNN for Sine Prediction

We start with a simple example: predicting future values of a sine function. This is an ideal task for understanding how RNNs work.

### Generate Data

In [ ]:
def generate_sine_wave(seq_length=100, num_sequences=1000):
    # Generate random sine wave sequences
    X = []
    y = []

    for _ in range(num_sequences):
        start = np.random.uniform(0, 10)
        freq = np.random.uniform(0.5, 2.0)

        t = np.linspace(start, start + 4 * np.pi, seq_length + 1)
        sine_wave = np.sin(freq * t)

        X.append(sine_wave[:-1])
        y.append(sine_wave[-1])

    X = np.array(X).reshape(-1, seq_length, 1).astype("float32")
    y = np.array(y).reshape(-1, 1).astype("float32")

    return X, y


X_train, y_train = generate_sine_wave(seq_length=50, num_sequences=800)
X_test, y_test = generate_sine_wave(seq_length=50, num_sequences=200)

print(f"Training data shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test data shape: X={X_test.shape}, y={y_test.shape}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for i in range(4):
    axes[i].plot(X_train[i], label="Input sequence")
    axes[i].axhline(
        y=y_train[i], color="r", linestyle="--", label=f"Target: {y_train[i][0]:.3f}"
    )
    axes[i].set_xlabel("Time step")
    axes[i].set_ylabel("Value")
    axes[i].set_title(f"Example sequence {i + 1}")
    axes[i].legend()
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Implementation of a Simple RNN

A `SimpleRNN` layer applies the recurrence above at every step and returns the hidden state. We set `return_sequences=False` so it returns only the final state, which is what a single-value prediction needs. Setting it to `True` returns the state at every step, which you want when stacking recurrent layers or predicting one value per step.

A `Dense` layer then maps that final state to a single number.

The alternative would be to flatten the whole sequence and feed it to a plain `Dense` network. That works, but it treats each position as an independent feature: it learns nothing about order, and a model trained on 50 steps cannot accept 60. The recurrence is what makes the model reusable across lengths.

In [ ]:
def build_simple_rnn(input_shape, hidden_size=32, output_size=1):

    model = models.Sequential(
        [
            layers.Input(shape=input_shape),
            layers.SimpleRNN(hidden_size, return_sequences=False),
            layers.Dense(output_size),
        ]
    )

    return model


model_rnn = build_simple_rnn(input_shape=(50, 1), hidden_size=32, output_size=1)

model_rnn.compile(
    optimizer=optimizers.Adam(learning_rate=0.01), loss="mse", metrics=["mae"]
)

print(model_rnn.summary())
print(f"\nNumber of parameters: {model_rnn.count_params()}")

### Training the RNN

In [ ]:
history_rnn = model_rnn.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1,
)

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history_rnn.history["loss"], label="Training loss", alpha=0.7)
plt.plot(history_rnn.history["val_loss"], label="Validation loss", alpha=0.7)
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Training and validation loss over epochs (Simple RNN)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Both curves fall steeply and then flatten, and the validation loss stays close to the training loss. The model is learning the shape of the wave rather than memorising individual training sequences.

In [ ]:
predictions = model_rnn.predict(X_test)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for i in range(4):
    idx = i * 10
    axes[i].plot(X_test[idx], label="Input sequence", alpha=0.7)
    axes[i].axhline(
        y=y_test[idx], color="g", linestyle="--", label=f"Actual: {y_test[idx][0]:.3f}"
    )
    axes[i].axhline(
        y=predictions[idx],
        color="r",
        linestyle="--",
        label=f"Prediction: {predictions[idx][0]:.3f}",
    )
    axes[i].set_xlabel("Time step")
    axes[i].set_ylabel("Value")
    axes[i].set_title(f"Prediction example {i + 1}")
    axes[i].legend()
    axes[i].grid(alpha=0.3)

plt.tight_layout()
plt.show()

mae = np.mean(np.abs(predictions - y_test))
print(f"\nMean Absolute Error: {mae:.4f}")

The model predicts the next value of each sine wave with a mean absolute error of about 0.006. The targets lie between -1 and 1, so the error is well under one percent of the range, and in the plots the predicted line sits almost on top of the actual value.

This task is easy for a recurrent network because the information it needs is recent: the last few points of a sine wave already determine the next one, so the hidden state only has to carry information a short distance. Part 2 asks what happens when the useful information is much further back.

## Part 2: The Vanishing Gradient Problem

Training an RNN uses backpropagation through time. To update the weights from a loss computed at the last step, the gradient has to travel back through every earlier step, and each step back multiplies it by the same recurrent Jacobian. That Jacobian is dominated by the recurrent weight matrix $W_{hh}$.

After $t$ steps the gradient has been multiplied by $W_{hh}$ about $t$ times, so its size is governed by the largest absolute eigenvalue of $W_{hh}$, known as the spectral radius $\rho$:

* $\rho < 1$: the gradient shrinks geometrically, which is the vanishing gradient problem
* $\rho > 1$: the gradient grows geometrically, which is the exploding gradient problem
* $\rho = 1$: a knife-edge case that preserves the signal but that no optimiser can hold exactly

We can measure this decay directly rather than inferring it from a training run. Whether a model trains well depends on initialisation, learning rate and luck, but this matrix product does not: it is the same every time you run it.

In [ ]:
# Measure how much of the gradient survives after t steps back through time.
# The same random matrix is rescaled to each spectral radius, so the radius is
# the only thing that differs between the curves.


def gradient_factor_over_time(spectral_radius, steps=100, hidden_size=32, seed=RSEED):
    rng = np.random.default_rng(seed)
    W = rng.standard_normal((hidden_size, hidden_size))
    W = W / max(abs(np.linalg.eigvals(W))) * spectral_radius

    product = np.eye(hidden_size)
    factors = []
    for _ in range(steps):
        product = product @ W
        factors.append(np.linalg.norm(product))

    return np.array(factors)


radii = [0.8, 0.95, 1.0, 1.05]
decay = {r: gradient_factor_over_time(r) for r in radii}

for r, factors in decay.items():
    print(f"Spectral radius {r}: gradient factor after 100 steps = {factors[-1]:.3e}")

In [ ]:
plt.figure(figsize=(10, 6))

for r, factors in decay.items():
    plt.plot(
        range(1, len(factors) + 1), factors, label=f"Spectral radius {r}", linewidth=2
    )

plt.axhline(y=1.0, color="grey", linestyle="--", alpha=0.6)
plt.yscale("log")
plt.xlabel("Steps back through time")
plt.ylabel("Gradient factor (log scale)")
plt.title("How the gradient scales as it travels back through a simple RNN")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The numbers show how quickly the gradient disappears. With a spectral radius of 0.8, the gradient arriving at the first step is 6.2e-09 of its original size, so a dependency 100 steps back is invisible to the optimiser: the weights that would capture it receive effectively no update. At 0.95 the factor is 0.18, which still carries signal. At 1.05 it reaches 4.0e+03, the exploding gradient case, which in practice shows up as a loss that jumps to NaN.

Only a radius of exactly 1.0 keeps the factor in a workable range (30.5), and no optimiser can pin a weight matrix to that value while it is learning. Note that the decay is geometric, so the curve is a straight line on a logarithmic axis: doubling the distance back in time squares the shrinkage.

This is why a simple RNN struggles with long-range dependencies. The problem is structural, not a matter of training for longer.

## Part 3: How an LSTM Preserves the Gradient

An LSTM carries a separate cell state $c_t$ alongside the hidden state, and updates it additively under the control of a forget gate $f_t$:

$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$

Along that cell-state path the derivative from one step to the next is just $f_t$, so the factor over 100 steps is the product of the forget gates rather than a product of weight matrices.

That difference is the whole idea. In a simple RNN the shrinkage is fixed by a weight matrix that also has to do the learning, so it cannot be tuned independently. In an LSTM the network can learn to hold the gate open for information worth keeping, and close it for information worth discarding, per step and per feature.

```mermaid
flowchart LR
    cprev[Cell state at t-1] --> keep[Multiply by forget gate]
    keep --> combine[Add new candidate]
    combine --> cnow[Cell state at t]
    forget[Forget gate] --> keep
    inp[Input gate] --> combine
    cand[Candidate value] --> combine
```

The path along the top of that diagram is the one that matters for gradients: it is addition and scaling, with no weight matrix in the way.

In [ ]:
# The gradient factor along the LSTM cell state is the product of the forget gates.

forget_gates = [0.5, 0.9, 0.99, 1.0]
steps = 100

for f in forget_gates:
    print(
        f"Forget gate held at {f}: gradient factor after {steps} steps = {f**steps:.3e}"
    )

In [ ]:
timesteps = np.arange(1, 101)

plt.figure(figsize=(10, 6))

for f in forget_gates:
    plt.plot(timesteps, f**timesteps, label=f"LSTM, forget gate {f}", linewidth=2)

plt.plot(
    timesteps,
    decay[0.8],
    label="Simple RNN, spectral radius 0.8",
    linestyle="--",
    color="black",
    linewidth=2,
)

plt.axhline(y=1.0, color="grey", linestyle="--", alpha=0.6)
plt.yscale("log")
plt.xlabel("Steps back through time")
plt.ylabel("Gradient factor (log scale)")
plt.title("Gradient flow: LSTM cell state versus a simple RNN")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

A gate at 0.5 is worse than the simple RNN: the factor falls to 7.9e-31, because halving a value 100 times destroys it. A gate at 0.9 gives 2.7e-05, at 0.99 it gives 0.37, and a gate held fully open at 1.0 gives exactly 1.0, meaning the gradient reaches the first step undiminished.

The lesson is not that an LSTM automatically has better gradient flow. A freshly initialised LSTM does not: its forget gate sits near the middle of its range, which is the 0.5 row above. The advantage is that the gate is *learnable*, so training can push it towards 1.0 for whatever the model needs to remember. A simple RNN has no equivalent control, since the same matrix that carries the gradient is also the matrix that must fit the data.

Below we build the LSTM and inspect its structure.

In [ ]:
def build_lstm_model(input_shape, hidden_size=32, num_layers=1, output_size=1):

    model = models.Sequential()

    model.add(layers.Input(shape=input_shape))
    model.add(layers.LSTM(hidden_size, return_sequences=(num_layers > 1)))

    for i in range(1, num_layers):
        return_seq = i < num_layers - 1
        model.add(layers.LSTM(hidden_size, return_sequences=return_seq))

    model.add(layers.Dense(output_size))

    return model


model_lstm = build_lstm_model(
    input_shape=(100, 1), hidden_size=32, num_layers=1, output_size=1
)
model_lstm.compile(
    optimizer=optimizers.Adam(learning_rate=0.01), loss="mse", metrics=["mae"]
)

print(model_lstm.summary())

## Part 4: GRU - Alternative to LSTM

In [ ]:
def build_gru_model(input_shape, hidden_size=32, num_layers=1, output_size=1):

    model = models.Sequential()

    model.add(layers.Input(shape=input_shape))
    model.add(layers.GRU(hidden_size, return_sequences=(num_layers > 1)))

    for i in range(1, num_layers):
        return_seq = i < num_layers - 1
        model.add(layers.GRU(hidden_size, return_sequences=return_seq))

    model.add(layers.Dense(output_size))

    return model


model_gru = build_gru_model(
    input_shape=(100, 1), hidden_size=32, num_layers=1, output_size=1
)
model_gru.compile(
    optimizer=optimizers.Adam(learning_rate=0.01), loss="mse", metrics=["mae"]
)

print(model_gru.summary())

In [ ]:
# Gating is not free. Compare the parameter counts at the same hidden size.

print("Parameters at hidden size 32, one output unit:")
print(f"Simple RNN: {model_rnn.count_params():>6}")
print(f"GRU:        {model_gru.count_params():>6}")
print(f"LSTM:       {model_lstm.count_params():>6}")

The simple RNN uses 1,121 parameters, the GRU 3,393 and the LSTM 4,385, so gating costs roughly three to four times as many parameters for the same 32 hidden units. A GRU sits between the two because it uses two gates rather than three and merges the cell state into the hidden state.

That is the trade-off in practice: a GRU often matches an LSTM at lower cost, so it is worth trying both rather than assuming the LSTM wins.

> Note that the small synthetic tasks in this notebook are not enough to separate these three architectures by accuracy. Their measured advantage appears on real sequence data, with longer training and enough examples to learn a long-range dependency. What this notebook shows is the mechanism that explains why that advantage exists.

## Part 5: Practical Application - Time Series Forecasting

Now we apply our knowledge to a more realistic task: forecasting passenger numbers in a time series.

The series below combines three ingredients: a slow upward trend, a seasonal cycle that repeats every 50 steps, and random noise. We normalise it before training, because the raw values sit around 100 and inputs that large push a recurrent layer's activations into saturation, where the gradients are small.

We give the model a window of 20 consecutive points and ask for the next one. An alternative is to predict several steps ahead at once, which avoids feeding predictions back into the model but needs more output units and more data to fit them.

In [ ]:
def create_sequences(data, seq_length):
    # Create sequences for time series
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i : i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)


np.random.seed(RSEED)
t = np.arange(500)
trend = 0.02 * t
seasonal = 10 * np.sin(2 * np.pi * t / 50)
noise = np.random.randn(500) * 2
passengers = 100 + trend + seasonal + noise

mean_pass = passengers.mean()
std_pass = passengers.std()
passengers_norm = (passengers - mean_pass) / std_pass

seq_length = 20
X_ts, y_ts = create_sequences(passengers_norm, seq_length)
X_ts = X_ts.reshape(-1, seq_length, 1).astype("float32")
y_ts = y_ts.reshape(-1, 1).astype("float32")

split = int(0.8 * len(X_ts))
X_train_ts, X_test_ts = X_ts[:split], X_ts[split:]
y_train_ts, y_test_ts = y_ts[:split], y_ts[split:]

print(f"Time series data: {X_train_ts.shape} / {X_test_ts.shape}")

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(t, passengers, label="Passengers", alpha=0.7)
plt.axvline(x=split + seq_length, color="r", linestyle="--", label="Train/test split")
plt.xlabel("Time")
plt.ylabel("Number of passengers")
plt.title("Passenger time series with trend and seasonality")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
model_ts = build_lstm_model(
    input_shape=(20, 1), hidden_size=64, num_layers=2, output_size=1
)
model_ts.compile(
    optimizer=optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"]
)

print("Training LSTM for time series forecasting:")
history_ts = model_ts.fit(
    X_train_ts,
    y_train_ts,
    epochs=100,
    batch_size=32,
    validation_data=(X_test_ts, y_test_ts),
    verbose=1,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    ],
)

In [ ]:
predictions_ts = model_ts.predict(X_test_ts)
y_test_denorm = y_test_ts * std_pass + mean_pass
predictions_denorm = predictions_ts * std_pass + mean_pass

test_indices = np.arange(split + seq_length, len(passengers))

plt.figure(figsize=(14, 6))
plt.plot(t, passengers, label="Actual data", alpha=0.7)
plt.plot(
    test_indices,
    predictions_denorm,
    label="LSTM predictions",
    color="r",
    alpha=0.7,
    linewidth=2,
)
plt.axvline(
    x=split + seq_length, color="g", linestyle="--", label="Test start", alpha=0.5
)
plt.xlabel("Time")
plt.ylabel("Number of passengers")
plt.title("LSTM time series forecasting")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

mae_ts = np.mean(np.abs(predictions_denorm - y_test_denorm))
rmse_ts = np.sqrt(np.mean((predictions_denorm - y_test_denorm) ** 2))

print(f"\nMean Absolute Error: {mae_ts:.2f}")
print(f"Root Mean Squared Error: {rmse_ts:.2f}")

The forecast follows both the seasonal cycle and the upward trend, with a mean absolute error of about 2.1 passengers and a root mean squared error of about 2.6.

Those figures only mean something with context. The series has a mean level of 105 and runs from 87 to 123, so the error is around two percent of the level. More usefully, the data was built with random noise of standard deviation 2.0, which nothing can predict, and that noise puts a floor of roughly 1.6 on the achievable MAE. Predicting the series mean every time would give an MAE of 6.75. The model is therefore close to the best score available and about three times better than the naive baseline.

Note also that RMSE (2.6) exceeds MAE (2.1). RMSE squares the errors before averaging, so it punishes occasional large misses more heavily. When the two diverge sharply, look for outliers in the predictions rather than a uniformly worse fit.

## Summary

In this notebook you:

- **Built and trained** SimpleRNN, LSTM and GRU models in TensorFlow/Keras, starting from a sine prediction task that the recurrence solves to a mean absolute error of about 0.006.
- **Measured** how the gradient scales as it travels back through time and found the decay is geometric: at a spectral radius of 0.8 only 6.2e-09 of the gradient survives 100 steps, while at 1.05 it explodes to 4.0e+03.
- **Compared** the three architectures by the control each gives over that decay, and by what the extra control costs in parameters.
- **Applied** an LSTM to a time series forecast, reaching a mean absolute error of about 2.1 against a noise floor near 1.6 and a mean-prediction baseline of 6.75.

| Architecture | Parameters at 32 units | Path the gradient takes |
| --- | --- | --- |
| SimpleRNN | 1,121 | Repeated multiplication by one weight matrix, decay fixed by its spectral radius |
| GRU | 3,393 | Two gates, update and reset, with the cell state merged into the hidden state |
| LSTM | 4,385 | Three gates on an additive cell state, so the decay is learnable per step |

**Important notes for practice:**

- Normalise your data before training
- Experiment with hidden size and number of layers
- Use early stopping to avoid overfitting
- For very long sequences (more than about 1000 steps), consider Transformers

## Exercises

1. **Experiment**: Train a bidirectional LSTM on the time series data. Compare the performance with the unidirectional LSTM.

2. **Hyperparameters**: Experiment with different values for `hidden_size` and `num_layers`. How does this affect performance and training time?

3. **Sequence length**: What happens when you change the sequence length in time series forecasting (e.g., to 5, 10, 50)? Is there an optimal value?

4. **Real-world application**: Download a real time series dataset (e.g., stock prices, temperature, etc.) and apply the techniques you learned.

## References & Further Reading

- [**Understanding LSTM Networks**](https://colah.github.io/posts/2015-08-Understanding-LSTMs/): Christopher Olah's walkthrough of the LSTM cell, with the clearest diagrams of the gates you will find anywhere.
- [**The Unreasonable Effectiveness of Recurrent Neural Networks**](https://karpathy.github.io/2015/05/21/rnn-effectiveness/): Andrej Karpathy trains character-level RNNs on Shakespeare and Linux source code, and inspects what the hidden state has learned.
- [**On the difficulty of training Recurrent Neural Networks**](https://arxiv.org/abs/1211.5063): Pascanu, Mikolov and Bengio, the paper behind the vanishing and exploding gradient analysis in Part 2.
- [**Working with RNNs**](https://www.tensorflow.org/guide/keras/working_with_rnns): the TensorFlow guide to recurrent layers, covering statefulness, masking and bidirectional wrappers.
- [**Keras LSTM layer**](https://keras.io/api/layers/recurrent_layers/lstm/): API reference for every argument used here.